# PullRequestReviewCommentEvent pipeline test (`06_pull_request_review_comment_events`)

This notebook tests `06_pull_request_review_comment_events` with a small synthetic Bronze table in the schema `github_lakehouse.test_review_comment`. It runs the real notebook with `%run` and `test_schema = test_review_comment`, so **production tables are never written**. The last check section proves that by comparing the production table versions before and after.

**Before you run it**, the production tables must exist, because the test tables are created with `CREATE TABLE ... LIKE`:

1. `11_silver_audit_table` (audit table; it already exists from the PushEvent work)
2. cells 1–6 of `06_pull_request_review_comment_events` (they create `silver.pull_request_review_comment_events` and `quarantine.quarantine_pull_request_review_comment_event`)

Every check prints `PASS` or `FAIL`. A section stops with an error if any of its checks failed. The last cell drops the test schema.

The shared pipeline (parse errors, core IDs, duplicates, all-dates mode, partial failure) is already covered by `tests/test_08_fork_events`. This test covers what is specific to PullRequestReviewCommentEvent: the four review-comment fields are read from `payload.pull_request` and `payload.comment` and converted, every problem in those fields is kept with a warning instead of quarantined, a missing PR object is counted in `null_pr_id_and_number`, an action other than `created` is kept with an `UNEXPECTED_VALUE` warning, two events about the same comment are both kept (dedup is by `event_id` only), and the quarantine table keeps the review-comment fields and their original text.

## B1. Setup: test schema, empty tables and helpers

In [0]:
import json
from datetime import date, datetime, timezone

TEST_SCHEMA = "test_review_comment"
assert TEST_SCHEMA.startswith("test_")   # never drop anything outside a test schema

def T(name):
    return f"github_lakehouse.{TEST_SCHEMA}.{name}"

TEST_BRONZE = T("github_events_raw")
TEST_SILVER = T("pull_request_review_comment_events")
TEST_QUARANTINE = T("quarantine_pull_request_review_comment_event")
TEST_AUDIT = T("silver_pipeline_runs")

PROD_TABLES = [
    "github_lakehouse.bronze.github_events_raw",
    "github_lakehouse.silver.pull_request_review_comment_events",
    "github_lakehouse.quarantine.quarantine_pull_request_review_comment_event",
    "github_lakehouse.audit.silver_pipeline_runs",
]

# The test tables copy the production definitions, so those must exist first.
missing = [t for t in PROD_TABLES if not spark.catalog.tableExists(t)]
if missing:
    raise RuntimeError(
        f"Missing production tables: {missing}. Run 11_silver_audit_table "
        "and cells 1-6 of 06_pull_request_review_comment_events (table creation) first."
    )

spark.sql(f"CREATE SCHEMA IF NOT EXISTS github_lakehouse.{TEST_SCHEMA}")
for t in [TEST_BRONZE, TEST_SILVER, TEST_QUARANTINE, TEST_AUDIT]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

# Bronze: only the four columns the pipeline reads.
spark.sql(f"""
    CREATE TABLE {TEST_BRONZE} (
        raw_json STRING, _source_file STRING,
        _source_date DATE, _ingested_at TIMESTAMP
    ) USING DELTA
""")
# Targets: exact copies of the production table definitions (no data).
spark.sql(f"CREATE TABLE {TEST_SILVER} LIKE github_lakehouse.silver.pull_request_review_comment_events")
spark.sql(f"CREATE TABLE {TEST_QUARANTINE} LIKE github_lakehouse.quarantine.quarantine_pull_request_review_comment_event")
spark.sql(f"CREATE TABLE {TEST_AUDIT} LIKE github_lakehouse.audit.silver_pipeline_runs")


# ---- assertion helpers ----
_results = []

def check(label, actual, expected):
    ok = actual == expected
    _results.append((label, ok))
    print(f"{'PASS' if ok else 'FAIL'}  {label}: actual={actual!r} expected={expected!r}")

def finish(section):
    failed = [label for label, ok in _results if not ok]
    _results.clear()
    if failed:
        raise AssertionError(f"{section}: {len(failed)} check(s) failed: {failed}")
    print(f"\n{section}: ALL CHECKS PASSED")

def check_audit(expected_counts, expected_quality=None,
                expected_status="SUCCEEDED", reconciled=True):
    # `audit` is the dict created by the pipeline's audit cell via %run.
    check("audit status", audit["status"], expected_status)
    check("audit reconciliation_passed", audit["reconciliation_passed"], reconciled)
    for k, v in expected_counts.items():
        check(f"audit {k}", audit.get(k), v)
    for k, v in (expected_quality or {}).items():
        check(f"quality {k}", audit["quality_metrics"].get(k), v)

def rows(table, where="true"):
    return [r.asDict() for r in spark.table(table).where(where).collect()]

def one_silver(event_id):
    found = rows(TEST_SILVER, f"event_id = '{event_id}'")
    check(f"silver has exactly one '{event_id}'", len(found), 1)
    return found[0] if found else {}

def table_version(t):
    return spark.sql(f"DESCRIBE HISTORY {t} LIMIT 1").collect()[0]["version"]

print("Setup complete:", TEST_BRONZE, TEST_SILVER, TEST_QUARANTINE, TEST_AUDIT)

## B2. Safety snapshot of the production tables

In [0]:
prod_versions_before = {t: table_version(t) for t in PROD_TABLES}
prod_versions_before

## B3. Load the synthetic Bronze data

The default event is a `created` inline comment `id 7001` on pull request `id 9001`, `number 42`.

| Row | Case | Expected result |
|---|---|---|
| `c1` | clean PullRequestReviewCommentEvent | Silver, every field converted, no warning |
| `c2` | `action` missing | Silver, `comment_action` NULL, warning `MISSING: comment_action` |
| `c3` | `action = edited` | Silver, value kept, warning `UNEXPECTED_VALUE: comment_action` |
| `c4` | `pull_request.id` missing, number present | Silver, `pr_id` NULL, `pr_number` kept, warning `MISSING: pr_id` |
| `c5` | `pull_request.number = "abc"` | Silver, `pr_number` NULL, warning `INVALID_CAST: pr_number` |
| `c6` | no `pull_request` object | Silver, two `MISSING` warnings, counted in `null_pr_id_and_number` |
| `c7` | `comment.id` missing | Silver, `comment_id` NULL, warning `MISSING: comment_id` |
| `c8` | `comment.id = "x9"` | Silver, `comment_id` NULL, warning `INVALID_CAST: comment_id` |
| `c9` | `pull_request` is text, not an object | quarantine `JSON_PARSE_ERROR` |
| `c10` | `repo.id` missing | quarantine `MISSING: repo_id`, review-comment fields and original text kept |
| `c11` (twice) | duplicate `event_id`, older and newer ingestion | one Silver row, from the newer file |
| `c12a`, `c12b` | two events with the same `comment_id` | both kept: dedup is by `event_id` only |
| `x1` | a PullRequestReviewEvent | ignored (not read by this notebook) |

In [0]:
F1 = "synthetic/2024-02-01-0.json.gz"
F2 = "synthetic/2024-02-01-1.json.gz"
D1 = date(2024, 2, 1)
I1 = datetime(2024, 3, 1, tzinfo=timezone.utc)   # earlier ingestion
I2 = datetime(2024, 3, 2, tzinfo=timezone.utc)   # later ingestion

MISSING = object()   # marker: leave the key out of the JSON

def payload(action="created", pull_request=None, comment=None):
    """A review-comment payload. Defaults = a clean, created comment."""
    p = {
        "action": action,
        "pull_request": {"id": 9001, "number": 42} if pull_request is None else pull_request,
        "comment": {"id": 7001} if comment is None else comment,
    }
    return {k: v for k, v in p.items() if v is not MISSING}

def ev(event_id, type_="PullRequestReviewCommentEvent", created_at="2024-02-01T10:00:00Z",
       actor=None, repo=None, payload_=None):
    """One GitHub-style event as a JSON string. Defaults = a clean event."""
    event = {
        "id": event_id,
        "type": type_,
        "created_at": created_at,
        "actor": {"id": 101, "login": "alice"} if actor is None else actor,
        "repo": {"id": 201, "name": "octo/repo"} if repo is None else repo,
        "payload": payload() if payload_ is None else payload_,
    }
    return json.dumps(event)

synthetic_rows = [
    # (raw_json, _source_file, _source_date, _ingested_at)                        # case
    (ev("c1"),                                                      F1, D1, I1),   # clean
    (ev("c2", payload_=payload(action=MISSING)),                    F1, D1, I1),   # action missing
    (ev("c3", payload_=payload(action="edited")),                   F1, D1, I1),   # unexpected action
    (ev("c4", payload_=payload(pull_request={"number": 42})),       F1, D1, I1),   # pr_id missing
    (ev("c5", payload_=payload(pull_request={"id": 9001, "number": "abc"})),
                                                                    F1, D1, I1),   # pr_number invalid
    (ev("c6", payload_=payload(pull_request=MISSING)),              F1, D1, I1),   # no pull_request
    (ev("c7", payload_=payload(comment={})),                        F1, D1, I1),   # comment_id missing
    (ev("c8", payload_=payload(comment={"id": "x9"})),              F1, D1, I1),   # comment_id invalid
    (ev("c9", payload_=payload(pull_request="oops")),               F1, D1, I1),   # parse error
    (ev("c10", repo={"name": "octo/repo"}),                         F1, D1, I1),   # repo_id missing
    (ev("c11", payload_=payload(comment={"id": 1101})),             F1, D1, I1),   # duplicate, older
    (ev("c11", payload_=payload(comment={"id": 1102})),             F2, D1, I2),   # duplicate, newer -> wins
    (ev("c12a", payload_=payload(comment={"id": 1200})),            F1, D1, I1),   # same comment_id ...
    (ev("c12b", payload_=payload(comment={"id": 1200})),            F1, D1, I1),   # ... different event
    (ev("x1", type_="PullRequestReviewEvent"),                      F1, D1, I1),   # other event type
]

(spark.createDataFrame(synthetic_rows, spark.table(TEST_BRONZE).schema)
    .write.mode("overwrite").saveAsTable(TEST_BRONZE))

check("synthetic bronze rows", spark.table(TEST_BRONZE).count(), 15)
finish("B3 load")

## B4. Run 1: single date 2024-02-01

In [0]:
%run ../06_pull_request_review_comment_events $test_schema="test_review_comment" $source_date="2024-02-01" $run_id="test-run1"

## B5. Run 1 checks

In [0]:
QUALITY_RUN1 = {
    "null_event_timestamp": 0, "null_actor_id": 0, "null_actor_login": 0,
    "null_repo_name": 0,
    "null_comment_action": 1, "null_pr_id": 2, "null_pr_number": 2,
    "null_pr_id_and_number": 1, "null_comment_id": 2,
    "unexpected_comment_action": 1,
    "missing_lineage": 0,
}

# --- tables and audit identity: the run used the test tables only
check("resolved silver", SILVER_TABLE, TEST_SILVER)
check("resolved quarantine", QUARANTINE_TABLE, TEST_QUARANTINE)
check("audit pipeline_name", audit["pipeline_name"], "silver_pull_request_review_comment_events")
check("audit event_type", audit["event_type"], "PullRequestReviewCommentEvent")

# --- audit: 14 PullRequestReviewCommentEvent rows (x1 is another type)
check_audit({
    "bronze_input_rows": 14, "hard_null_quarantine_rows": 2,
    "validation_quarantine_rows": 0, "validated_rows": 12,
    "duplicate_rows_removed": 1, "silver_candidate_rows": 11,
    "silver_rows_inserted": 11, "silver_rows_already_present": 0,
    "quarantine_candidate_rows": 2, "quarantine_duplicates_removed": 0,
    "quarantine_rows_inserted": 2,
}, QUALITY_RUN1)
check("quality metric names", sorted(audit["quality_metrics"]), sorted(QUALITY_RUN1))

# --- Silver schema: the contract columns, event fields before lineage
check("silver columns", spark.table(TEST_SILVER).columns, [
    "event_id", "event_type", "event_timestamp", "event_date",
    "actor_id", "actor_login", "repo_id", "repo_name",
    "comment_action", "pr_id", "pr_number", "comment_id",
    "_source_file", "_source_date", "_ingested_at", "quarantine_warning",
])

# --- Silver contents
check("silver event_ids",
      sorted(r["event_id"] for r in rows(TEST_SILVER)),
      sorted(["c1", "c2", "c3", "c4", "c5", "c6", "c7", "c8", "c11", "c12a", "c12b"]))

def comment_fields(r):
    return (r.get("comment_action"), r.get("pr_id"), r.get("pr_number"), r.get("comment_id"))

r = one_silver("c1")
check("c1 fields", comment_fields(r), ("created", 9001, 42, 7001))
check("c1 common fields", (r.get("actor_id"), r.get("repo_id"), str(r.get("event_date"))),
      (101, 201, "2024-02-01"))
check("c1 warning", r.get("quarantine_warning"), None)

r = one_silver("c2")
check("c2 action", r.get("comment_action"), None)
check("c2 warning", r.get("quarantine_warning"), "MISSING: comment_action")

r = one_silver("c3")
check("c3 action kept unchanged", r.get("comment_action"), "edited")
check("c3 warning", r.get("quarantine_warning"), "UNEXPECTED_VALUE: comment_action")

r = one_silver("c4")
check("c4 pr_id NULL, pr_number kept", (r.get("pr_id"), r.get("pr_number")), (None, 42))
check("c4 warning", r.get("quarantine_warning"), "MISSING: pr_id")

r = one_silver("c5")
check("c5 pr_number", r.get("pr_number"), None)
check("c5 warning", r.get("quarantine_warning"), "INVALID_CAST: pr_number")

r = one_silver("c6")
check("c6 fields", comment_fields(r), ("created", None, None, 7001))
check("c6 warnings", r.get("quarantine_warning"), "MISSING: pr_id; MISSING: pr_number")

r = one_silver("c7")
check("c7 comment_id", r.get("comment_id"), None)
check("c7 warning", r.get("quarantine_warning"), "MISSING: comment_id")

r = one_silver("c8")
check("c8 comment_id", r.get("comment_id"), None)
check("c8 warning", r.get("quarantine_warning"), "INVALID_CAST: comment_id")

r = one_silver("c11")
check("c11 winner = newer ingestion", (r.get("comment_id"), r.get("_source_file")), (1102, F2))

check("c12 both events kept for one comment_id",
      sorted(r["event_id"] for r in rows(TEST_SILVER, "comment_id = 1200")), ["c12a", "c12b"])

# --- Quarantine contents
q = rows(TEST_QUARANTINE)
check("quarantine reasons",
      sorted(r["quarantine_reason"] for r in q), ["JSON_PARSE_ERROR", "MISSING: repo_id"])
# A parse-error row is identified by its raw_json, because its parsed fields
# (including event_id) may be NULL.
parse_error = [r for r in q if r["quarantine_reason"] == "JSON_PARSE_ERROR"]
check("c9 is the parse error",
      [json.loads(r["raw_json"])["id"] for r in parse_error], ["c9"])
c10 = [r for r in q if r["event_id"] == "c10"]
check("c10 found once", len(c10), 1)
r = c10[0] if c10 else {}
check("c10 reason", r.get("quarantine_reason"), "MISSING: repo_id")
check("c10 fields kept", comment_fields(r), ("created", 9001, 42, 7001))
check("c10 original text kept",
      (r.get("pr_id_raw"), r.get("pr_number_raw"), r.get("comment_id_raw"), r.get("repo_id_raw")),
      ("9001", "42", "7001", None))
check("quarantine keeps raw_json", all(r["raw_json"] for r in q), True)
check("quarantine keys non-NULL", all(r["quarantine_key"] for r in q), True)

finish("B5 run 1")

## B6. Run 2: rerun the same date

In [0]:
%run ../06_pull_request_review_comment_events $test_schema="test_review_comment" $source_date="2024-02-01" $run_id="test-run2"

## B7. Run 2 checks: idempotency, plus the final safety check

In [0]:
check_audit({
    "bronze_input_rows": 14, "silver_candidate_rows": 11,
    "silver_rows_inserted": 0, "silver_rows_already_present": 11,
    "quarantine_candidate_rows": 2, "quarantine_rows_inserted": 0,
}, QUALITY_RUN1)
check("silver rows unchanged", spark.table(TEST_SILVER).count(), 11)
check("quarantine rows unchanged", spark.table(TEST_QUARANTINE).count(), 2)

statuses = [r["status"] for r in
            spark.table(TEST_AUDIT).orderBy("run_finished_at").collect()]
check("audit history", statuses, ["SUCCEEDED", "SUCCEEDED"])

for t in PROD_TABLES:
    check(f"prod version unchanged: {t}", table_version(t), prod_versions_before[t])
finish("B7 run 2 + production safety")

## Clean up: drop the test schema

In [0]:
assert TEST_SCHEMA.startswith("test_")
spark.sql(f"DROP SCHEMA IF EXISTS github_lakehouse.{TEST_SCHEMA} CASCADE")